In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

from io_utils import load_json
from model import build_ann_model   # arsitektur di-reuse, TIDAK ditulis ulang

print('TensorFlow version:', tf.__version__)
tf.random.set_seed(42)

TensorFlow version: 2.21.0


In [2]:
selected_features = load_json('selected_features_v2.json')
label_names = load_json('label_names_v2.json')

input_dim = len(selected_features)
n_classes = len(label_names)

print(f'Jumlah fitur input : {input_dim}')   # 34 fitur hasil RF importance
print(f'Jumlah kelas output: {n_classes} -> {label_names}')

Jumlah fitur input : 34
Jumlah kelas output: 3 -> {'0': 'Prime', '1': 'Performing', '2': 'Non-Performing'}


In [3]:
# ================================================
# Skenario eksperimen v2: E1-E6 (identik konsepnya dengan
# pipeline 15 fitur) + E7 = eksperimen SMOTE baru
# ================================================
EXPERIMENTS_V2 = {
    'E1': {'description': 'Baseline ANN (34 fitur)',                    'use_dropout': False, 'use_batchnorm': False, 'use_class_weight': False, 'use_smote': False},
    'E2': {'description': 'ANN + Dropout',                              'use_dropout': True,  'use_batchnorm': False, 'use_class_weight': False, 'use_smote': False},
    'E3': {'description': 'ANN + Batch Normalization',                  'use_dropout': False, 'use_batchnorm': True,  'use_class_weight': False, 'use_smote': False},
    'E4': {'description': 'ANN + Dropout + Batch Normalization',        'use_dropout': True,  'use_batchnorm': True,  'use_class_weight': False, 'use_smote': False},
    'E5': {'description': 'ANN + Class Weight',                        'use_dropout': False, 'use_batchnorm': False, 'use_class_weight': True,  'use_smote': False},
    'E6': {'description': 'ANN + Dropout + Class Weight',               'use_dropout': True,  'use_batchnorm': False, 'use_class_weight': True,  'use_smote': False},
    'E7': {'description': 'ANN + SMOTE (arsitektur sama seperti E1)',   'use_dropout': False, 'use_batchnorm': False, 'use_class_weight': False, 'use_smote': True},
}

In [4]:
models_by_experiment = {}
for exp_id, config in EXPERIMENTS_V2.items():
    models_by_experiment[exp_id] = build_ann_model(
        input_dim=input_dim,
        n_classes=n_classes,
        use_dropout=config['use_dropout'],
        use_batchnorm=config['use_batchnorm'],
        name=f'ann_v2_{exp_id.lower()}',
    )

for exp_id, model in models_by_experiment.items():
    print('=' * 80)
    print(f"{exp_id}: {EXPERIMENTS_V2[exp_id]['description']}")
    print('=' * 80)
    model.summary()

E1: Baseline ANN (34 fitur)


Model: "ann_v2_e1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 34)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │         8,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,771 (206.14 KB)

 Trainable params: 52,771 (206.14 KB)

...

Model: "ann_v2_e7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 34)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │         8,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,771 (206.14 KB)

 Trainable params: 52,771 (206.14 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
%%writefile ../src/model_v2.py
"""
src/model_v2.py — daftar eksperimen untuk pipeline 34 fitur (jalur 'b').
Arsitektur ANN tetap reuse dari model.build_ann_model (TIDAK diduplikasi).
"""

EXPERIMENTS_V2 = {
    'E1': {'description': 'Baseline ANN (34 fitur)',                  'use_dropout': False, 'use_batchnorm': False, 'use_class_weight': False, 'use_smote': False},
    'E2': {'description': 'ANN + Dropout',                            'use_dropout': True,  'use_batchnorm': False, 'use_class_weight': False, 'use_smote': False},
    'E3': {'description': 'ANN + Batch Normalization',                'use_dropout': False, 'use_batchnorm': True,  'use_class_weight': False, 'use_smote': False},
    'E4': {'description': 'ANN + Dropout + Batch Normalization',      'use_dropout': True,  'use_batchnorm': True,  'use_class_weight': False, 'use_smote': False},
    'E5': {'description': 'ANN + Class Weight',                      'use_dropout': False, 'use_batchnorm': False, 'use_class_weight': True,  'use_smote': False},
    'E6': {'description': 'ANN + Dropout + Class Weight',             'use_dropout': True,  'use_batchnorm': False, 'use_class_weight': True,  'use_smote': False},
    'E7': {'description': 'ANN + SMOTE (arsitektur sama seperti E1)', 'use_dropout': False, 'use_batchnorm': False, 'use_class_weight': False, 'use_smote': True},
}

Writing ../src/model_v2.py
